## imports

### reload

In [1]:
%load_ext autoreload
%autoreload 2

### imports dependencies

In [2]:
import json
import sys
import os
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

# ingestion
from app.ingestion.pipeline import chunk_ingestion_pipeline, question_ingestion_pipeline
from app.schemas import DocumentEval

# rag 
from app.rag.pipeline import rag_pipeline

# storage 
from app.storage.setup import setup_chunk_storage, setup_question_storage

# langchain
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings

# dotenv 
from dotenv import load_dotenv

# visualization
from tqdm import tqdm

# util
from app.util.time_util import current_timestamp_str

# config 
from app.config import EMBEDDING_MODEL

In [3]:
with open("../dataset/machine_learning_knowledge.json") as f:
    json_docs = json.load(f)
    print(f"docs: {json_docs[0]}")

with open("../dataset/machine_learning_knowledge_question_doc_mapping.json") as f:
    json_question_doc_mapping = json.load(f)
    print(f"docs_question: {json_question_doc_mapping[0]}")

docs: {'id': 'doc1', 'text': 'Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed. Algorithms analyze historical data and use it to make predictions or decisions. It is widely used in applications such as recommendation systems, fraud detection, and image recognition. The performance of a machine learning model typically improves as it is exposed to more high-quality data.', 'questions': ['What does machine learning mean and what is it used for?', 'How would you define machine learning and its primary purpose?', 'What is the role of machine learning in artificial intelligence?']}
docs_question: {'id': 'q1', 'question': 'What does machine learning mean and what is it used for?', 'docs': ['doc1', 'doc2', 'doc4', 'doc9']}


In [4]:
doc_source = [Document(metadata={"id": doc["id"]}, page_content=doc["text"]) for doc in json_docs]
doc_question = [Document(metadata={"docs": json.dumps(doc["docs"]), "id":doc["id"]}, page_content=doc["question"]) for doc in json_question_doc_mapping]

print(doc_source)
print(doc_question)

[Document(metadata={'id': 'doc1'}, page_content='Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed. Algorithms analyze historical data and use it to make predictions or decisions. It is widely used in applications such as recommendation systems, fraud detection, and image recognition. The performance of a machine learning model typically improves as it is exposed to more high-quality data.'), Document(metadata={'id': 'doc2'}, page_content='Supervised learning is a type of machine learning where models are trained using labeled data. Each training example contains both input features and the correct output. The algorithm learns a mapping from inputs to outputs so it can make predictions on new data. Common supervised tasks include classification and regression.'), Document(metadata={'id': 'doc3'}, page_content='Unsupervised learning involves training models on data that does not conta

### setup storage

In [5]:
embedding_function = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore_chunk = setup_chunk_storage(embedding_function)
vectorstore_question = setup_question_storage(embedding_function)

C:\Users\april\AppData\Local\Temp\ipykernel_13424\3032895038.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
D:\07_learning directory\02_self_learning\rag_retrieval_augmented_retrieval\code_ui\app\storage\setup.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(
d:\Programs\anaconda3\envs\rag_py3.10\lib\site-packages\opentelemetry\proto\collector\logs\v1\logs_service_pb2_grpc.py:21: RuntimeWarning: The grpc package installed is at version 1.60.1, but the generated code in opentelemetry/proto/collector/logs/v1/l

### setup dotenv

In [6]:
load_dotenv(dotenv_path='../.env')

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

## ingest

In [7]:
question_ingestion_pipeline(vectorstore_question, doc_question)
chunk_ingestion_pipeline(vectorstore_chunk, doc_source)

doc_question size: 12
collection size: 12
doc_source size: 20; chunked_docs size: 21
collection size: 58


[Document(metadata={'id': 'doc1', 'chunk_id': 'doc1_chunk0'}, page_content='Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed. Algorithms analyze historical data and use it to make predictions or decisions. It is widely used in applications such as recommendation systems, fraud detection, and image recognition'),
 Document(metadata={'id': 'doc1', 'chunk_id': 'doc1_chunk1'}, page_content='. The performance of a machine learning model typically improves as it is exposed to more high-quality data.'),
 Document(metadata={'id': 'doc2', 'chunk_id': 'doc2_chunk0'}, page_content='Supervised learning is a type of machine learning where models are trained using labeled data. Each training example contains both input features and the correct output. The algorithm learns a mapping from inputs to outputs so it can make predictions on new data. Common supervised tasks include classification and re

## rag pipeline

In [8]:
question = "What is the main goal of unsupervised learning?"

In [9]:
answer, contexts, system_prompt, user_prompt = rag_pipeline(vectorstore_chunk, vectorstore_question, question)
print(f"system_prompt: {system_prompt}")
print("-----")
print(f"user_prompt: {user_prompt}")


ResponseError: model requires more system memory (8.3 GiB) than is available (6.7 GiB) (status code: 500)

In [ ]:
print("===== ANSWER =====")
print(answer)
print("--------")
print("===== CONTEXT =====")
print(contexts)

===== ANSWER =====
The main goal of unsupervised learning is to discover hidden patterns or structures within the dataset, using techniques such as clustering and dimensionality reduction. The goal is to reduce overfitting by balancing the learning patterns while maintaining generalization. Techniques such as regularization, dropout, and cross-validation help reduce overfitting. Clustering is an unsupervised technique used to group similar data points together, which can be useful for customer segmentation and anomaly detection.
--------
===== CONTEXT =====
[Document(metadata={'chunk_id': 'doc3_chunk0', 'id': 'doc3'}, page_content='Unsupervised learning involves training models on data that does not contain labels. The goal is to discover hidden patterns or structures within the dataset. Techniques such as clustering and dimensionality reduction are commonly used. These methods are helpful for exploratory data analysis and feature discovery.'), Document(metadata={'id': 'doc3', 'chunk_i

In [ ]:
# search_result = vectorstore.similarity_search_with_score(question, k=5)
# print(search_result)

## evaluation

In [ ]:
with open("../dataset/machine_learning_knowledge_evaluation.json") as f:
    doc_eval_json = json.load(f)
    print(doc_eval_json)

[{'question': 'What is machine learning and what is its main goal?', 'answer': 'Machine learning is a field of artificial intelligence that enables computers to learn patterns from data and make predictions or decisions without explicit programming.', 'relevant_doc_ids': ['doc1']}, {'question': 'Give two examples of applications where machine learning is commonly used.', 'answer': 'Machine learning is commonly used in applications such as recommendation systems, fraud detection, and image recognition.', 'relevant_doc_ids': ['doc1']}, {'question': 'What type of machine learning uses labeled data for training?', 'answer': 'Supervised learning uses labeled data where each training example includes input features and the correct output.', 'relevant_doc_ids': ['doc2']}, {'question': 'What are two common tasks in supervised learning?', 'answer': 'Two common supervised learning tasks are classification and regression.', 'relevant_doc_ids': ['doc2', 'doc8', 'doc9']}, {'question': 'What is the 

In [ ]:
doc_eval = [DocumentEval(
                question=doc['question'], 
                answer=doc['answer'], 
                relevant_doc_ids=doc['relevant_doc_ids']) 
            for doc in doc_eval_json]

In [ ]:
for doc in doc_eval[:3]:
    print(doc.question)
    print(doc.answer)
    print(doc.relevant_doc_ids)
    print("----------")

What is machine learning and what is its main goal?
Machine learning is a field of artificial intelligence that enables computers to learn patterns from data and make predictions or decisions without explicit programming.
['doc1']
----------
Give two examples of applications where machine learning is commonly used.
Machine learning is commonly used in applications such as recommendation systems, fraud detection, and image recognition.
['doc1']
----------
What type of machine learning uses labeled data for training?
Supervised learning uses labeled data where each training example includes input features and the correct output.
['doc2']
----------


### run all cases

In [ ]:
answers = []
user_prompts = []
retrieved_contexts = []

for doc in tqdm(doc_eval): 
    answer, contexts, _, user_prompt = rag_pipeline(vectorstore_chunk, vectorstore_question, doc.question)
    if answer is None:
        answers.append("I don't know")
    else:
        answers.append(answer)

    user_prompts.append(user_prompt)
    retrieved_contexts.append(contexts)

100%|██████████| 27/27 [02:24<00:00,  5.35s/it]


### recall@k

In [ ]:
k = 5 

total = 0.0
valid_cases = 0

def recall_at_k(relevant, retrieved, k):
    if not relevant:
        return None, set()

    if not retrieved:
        return 0.0, set()

    retrieved_k = retrieved[:k]

    relevant_set = set(relevant)
    retrieved_set = set(retrieved_k)

    relevant_found = relevant_set.intersection(retrieved_set)

    recall = len(relevant_found) / len(relevant_set)

    return recall, relevant_found

for answer, retrieved, eval in tqdm(zip(answers, retrieved_contexts, doc_eval)):
    relevant = eval.relevant_doc_ids

    print(f"answer\t\t: {answer}")
    print(f"relevant\t: {relevant}")

    # get retrieved
    if retrieved is None:
        score = 0.0
        print("retrieved\t: None")
    else:
        retrieved_doc_ids = [r.metadata['id'] for r in retrieved]
        retrieved_doc_ids = list(dict.fromkeys(retrieved_doc_ids))
        print(f"retrieved_doc_ids: {retrieved_doc_ids}")

        score, relevant_found = recall_at_k(relevant, retrieved_doc_ids, k)
    
    print(f"recall@{k} score\t: {score}")
    print("=========================================")

    if score is not None:
        total += score
        valid_cases += 1


average = total / valid_cases if valid_cases > 0 else 0
print(f"average score: {average:.2f}")

27it [00:00, 17446.65it/s]

answer		: Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed. It involves using algorithms to analyze historical data and use it to make predictions or decisions, making it widely used in applications such as recommendation systems, fraud detection, and image recognition. Supervised learning is a type of machine learning where models are trained using labeled data, where each training example contains both input features and the correct output. The algorithm learns a mapping from inputs to outputs so it can make predictions on new data. Common supervised tasks include classification and regression.
relevant	: ['doc1']
retrieved_doc_ids: ['doc1', 'doc2', 'doc4', 'doc9']
recall@5 score	: 1.0
answer		: Clustering is an unsupervised learning technique used to group similar data points together, such as customer segmentation or anomaly detection in customer segmentation. It is often applie

### generation evaluation - ragas

In [ ]:
questions = [eval.question for eval in doc_eval]
ground_truths = [eval.answer for eval in doc_eval]

relevant_doc_ids   = [r.relevant_doc_ids for r in doc_eval]

contexts = []
for doc_list in retrieved_contexts:
    if doc_list is None:
        contexts.append([])
    else:
        contexts.append([doc.page_content for doc in doc_list])

print(contexts)
print(ground_truths)


[['Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed. Algorithms analyze historical data and use it to make predictions or decisions. It is widely used in applications such as recommendation systems, fraud detection, and image recognition', 'The performance of a machine learning model typically improves as it is exposed to more high-quality data.', 'Supervised learning is a type of machine learning where models are trained using labeled data. Each training example contains both input features and the correct output. The algorithm learns a mapping from inputs to outputs so it can make predictions on new data. Common supervised tasks include classification and regression.', 'Each training example contains both input features and the correct output. The algorithm learns a mapping from inputs to outputs so it can make predictions on new data', 'The algorithm learns a mapping from inputs 

In [ ]:
contexts[1]

['Unsupervised learning involves training models on data that does not contain labels. The goal is to discover hidden patterns or structures within the dataset. Techniques such as clustering and dimensionality reduction are commonly used. These methods are helpful for exploratory data analysis and feature discovery.',
 'The goal is to discover hidden patterns or structures within the dataset. Techniques such as clustering and dimensionality reduction are commonly used',
 'Techniques such as clustering and dimensionality reduction are commonly used. These methods are helpful for exploratory data analysis and feature discovery.',
 'Overfitting occurs when a machine learning model learns the training data too well, including its noise and irrelevant details. As a result, the model performs poorly on new, unseen data. Techniques such as regularization, dropout, and cross-validation help reduce overfitting. A good model balances learning patterns while maintaining generalization.',
 'As a r

In [ ]:
from datasets import Dataset

print(f"question len    : {len(questions)}")
print(f"contexts len    : {len(contexts)}")
print(f"response len    : {len(answers)}")
print(f"ground_truth len: {len(ground_truths)}")

data = {
    "question": questions,
    "contexts": contexts,
    "answer": answers,
    "ground_truth": ground_truths
}

dataset = Dataset.from_dict(data)

question len    : 27
contexts len    : 27
response len    : 27
ground_truth len: 27


In [ ]:
# dump all dataset
filename = f"ragas_dataset_{current_timestamp_str()}.json"
with open(filename, "w") as f:
    json.dump(data, f)
print(filename)

ragas_dataset_20260323_214754.json


In [ ]:
# import sys
# sys.exit()

In [ ]:
print('questions')
[print(q) for q in questions[:3]]
print('='*10)
print('answers')
[print(a+"\n-----") for a in answers[:3]]
print('='*10)
print('prompts')
[print(p+"\n-----") for p in user_prompts[:3]]

questions
What is machine learning and what is its main goal?
Give two examples of applications where machine learning is commonly used.
What type of machine learning uses labeled data for training?
answers
Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed. It involves using algorithms to analyze historical data and use it to make predictions or decisions, making it widely used in applications such as recommendation systems, fraud detection, and image recognition. Supervised learning is a type of machine learning where models are trained using labeled data, where each training example contains both input features and the correct output. The algorithm learns a mapping from inputs to outputs so it can make predictions on new data. Common supervised tasks include classification and regression.
-----
Clustering is an unsupervised learning technique used to group similar data points toget

[None, None, None]

In [ ]:
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from app.config import EVALUATION_MODEL

"""
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    temperature=0,
    model_kwargs={"n": 1}
)
"""

llm = ChatOpenAI(
    model=EVALUATION_MODEL,
    api_key=OPENAI_API_KEY,
    temperature=0,
    model_kwargs={"n": 1},
    timeout=600,
    max_retries=10
)

ragas_llm = LangchainLLMWrapper(llm)

d:\Programs\anaconda3\envs\rag_py3.10\lib\site-packages\IPython\core\interactiveshell.py:3519: UserWarning: Parameters {'n'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):
C:\Users\april\AppData\Local\Temp\ipykernel_7440\3792604272.py:24: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)


In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    answer_correctness,
    context_precision,
    context_recall
)

result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        # answer_relevancy,
        answer_correctness,
        # context_precision,
        context_recall
    ],
    llm=ragas_llm,
    embeddings=embedding_function
)

C:\Users\april\AppData\Local\Temp\ipykernel_7440\2749274894.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\april\AppData\Local\Temp\ipykernel_7440\2749274894.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\april\AppData\Local\Temp\ipykernel_7440\2749274894.py:2: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import (
C:\Users\april\AppData\Local\Temp\ipykernel_744

Evaluating:   0%|          | 0/81 [00:00<?, ?it/s]

In [ ]:
print(result)

df = result.to_pandas()
df.head()

{'faithfulness': 0.7016, 'answer_correctness': 0.6073, 'context_recall': 1.0000}


,user_input,retrieved_contexts,response,reference,faithfulness,answer_correctness,context_recall
0,What is machine learning and what is its main ...,[Machine learning is a field of artificial int...,Machine learning is a field of artificial inte...,Machine learning is a field of artificial inte...,1.0,0.492757,1.0
1,Give two examples of applications where machin...,[Unsupervised learning involves training model...,Clustering is an unsupervised learning techniq...,Machine learning is commonly used in applicati...,1.0,0.181005,1.0
2,What type of machine learning uses labeled dat...,[Machine learning is a field of artificial int...,Supervised learning is a type of machine learn...,Supervised learning uses labeled data where ea...,1.0,0.643615,1.0
3,What are two common tasks in supervised learning?,[Unsupervised learning involves training model...,Clustering and dimensionality reduction.,Two common supervised learning tasks are class...,0.0,0.175767,1.0
4,What is the main goal of unsupervised learning?,[Unsupervised learning involves training model...,The main goal of unsupervised learning is to d...,The goal of unsupervised learning is to discov...,0.5,0.520652,1.0


In [ ]:
filename = f'result_{current_timestamp_str()}.xlsx'
df.to_excel(filename)
print(filename)

result_20260323_220442.xlsx


## Sandbox

In [ ]:
answers[6]

'The three common dataset splits used in machine learning are the training set, validation set, and test set. The training set is used to teach the model patterns in the data, the validation set helps tune hyperparameters and prevent overfitting, and the test set evaluates how well the trained model generalizes to unseen data. Classification algorithms such as logistic regression, support vector machines, and random forests are commonly used. Other common dataset splits include k-means clustering and hierarchical clustering, which can be applied in customer segmentation and anomaly detection.'

In [ ]:
user_prompts[6]

'\nAnswer the following question based on given contexts:\n\nContext:\n- A dataset in machine learning is typically divided into training, validation, and test sets. The training set is used to teach the model patterns in the data. The validation set helps tune hyperparameters and prevent overfitting. The test set evaluates how well the trained model generalizes to unseen data.\n- The training set is used to teach the model patterns in the data. The validation set helps tune hyperparameters and prevent overfitting\n- The validation set helps tune hyperparameters and prevent overfitting. The test set evaluates how well the trained model generalizes to unseen data.\n- Classification is a machine learning task where the goal is to assign inputs into predefined categories. For example, an email classifier may determine whether a message is spam or not spam. Popular algorithms include logistic regression, support vector machines, and random forests. Accuracy, precision, recall, and F1-score

In [ ]:
print(question)

What is the main goal of unsupervised learning?


In [ ]:
from app.rag.pipeline import get_context_based_on_question, get_context_based_on_question_mapping
context_result = get_context_based_on_question(vectorstore_chunk, question)
context_result_question, q_results = get_context_based_on_question_mapping(vectorstore_chunk, vectorstore_question, question)
print(context_result)

[Document(metadata={'chunk_id': 'doc3_chunk0', 'id': 'doc3', 'distance': 0.24537262320518494}, page_content='Unsupervised learning involves training models on data that does not contain labels. The goal is to discover hidden patterns or structures within the dataset. Techniques such as clustering and dimensionality reduction are commonly used. These methods are helpful for exploratory data analysis and feature discovery.'), Document(metadata={'chunk_id': 'doc3_chunk1', 'id': 'doc3', 'distance': 0.4779125154018402}, page_content='. The goal is to discover hidden patterns or structures within the dataset. Techniques such as clustering and dimensionality reduction are commonly used')]


In [ ]:
context_result

[Document(metadata={'chunk_id': 'doc3_chunk0', 'id': 'doc3', 'distance': 0.24537262320518494}, page_content='Unsupervised learning involves training models on data that does not contain labels. The goal is to discover hidden patterns or structures within the dataset. Techniques such as clustering and dimensionality reduction are commonly used. These methods are helpful for exploratory data analysis and feature discovery.'),
 Document(metadata={'chunk_id': 'doc3_chunk1', 'id': 'doc3', 'distance': 0.4779125154018402}, page_content='. The goal is to discover hidden patterns or structures within the dataset. Techniques such as clustering and dimensionality reduction are commonly used')]

In [ ]:
context_result_question

[Document(metadata={'chunk_id': 'doc3_chunk0', 'id': 'doc3'}, page_content='Unsupervised learning involves training models on data that does not contain labels. The goal is to discover hidden patterns or structures within the dataset. Techniques such as clustering and dimensionality reduction are commonly used. These methods are helpful for exploratory data analysis and feature discovery.'),
 Document(metadata={'id': 'doc3', 'chunk_id': 'doc3_chunk1'}, page_content='. The goal is to discover hidden patterns or structures within the dataset. Techniques such as clustering and dimensionality reduction are commonly used'),
 Document(metadata={'chunk_id': 'doc3_chunk2', 'id': 'doc3'}, page_content='. Techniques such as clustering and dimensionality reduction are commonly used. These methods are helpful for exploratory data analysis and feature discovery.'),
 Document(metadata={'chunk_id': 'doc5_chunk0', 'id': 'doc5'}, page_content='Overfitting occurs when a machine learning model learns the

In [ ]:
q_results

[(Document(metadata={'docs': '["doc3", "doc15", "doc3", "doc10", "doc5"]', 'id': 'q6'}, page_content='Which techniques are typically applied in unsupervised learning tasks?'),
  0.3118478059768677)]

In [ ]:
len(context_result_question)

12